# A large dataset: 500,000 rows

The pattern for big datasets is: design once on a small run, look at the result, then scale the same schema up. The large run skips the design step, so it is fast and gives exactly the structure you approved.

**Uses:** 502,000 records (2,000 for the small run, 500,000 for the large one), no AI-generated fields. API reference: [generaterealdata.com/docs/api/](https://generaterealdata.com/docs/api/?ref=github).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://generaterealdata.com?ref=github), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. Design on a small run

In [2]:
def repo_file(path):
    """Read a file of this repository: from disk when the repo is cloned, from GitHub otherwise (e.g. in Colab)."""
    local = Path("..") / path
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get("https://raw.githubusercontent.com/generaterealdata/api-examples/main/" + path, timeout=30)
    r.raise_for_status()
    return r.text


SHOP_SQL = repo_file("schemas/shop.sql")
SHOP_RULES = repo_file("schemas/shop_rules.txt").strip()

small = generate({"mode": "sql", "sql": SHOP_SQL, "notes": SHOP_RULES, "rows": 2000, "format": "csv", "seed": 3})
summary(small)

{'status': 'ready',
 'format': 'csv',
 'rows_generated': 2000,
 'rows_per_table': {'products': 43,
  'order_items': 1276,
  'customers': 170,
  'orders': 511},
 'credits_charged': 2000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_cPMn19W8zl7KTZt0',
 'seed': 3}

## 2. Check what the large run needs

The small run returned a `schema_id`. A dry run with that schema and 500,000 rows confirms the request is valid and shows the records it will use, before anything is generated.

In [3]:
big_spec = {"mode": "schema", "schema_id": small["schema_id"], "rows": 500_000,
            "format": "sql", "dialect": "postgresql", "seed": 2026, "name": "Shop, 500K rows"}
check = api("POST", "/datasets", json={**big_spec, "dry_run": True})
{k: check[k] for k in ("valid", "rows", "credits_required", "ai_rows_required")}

{'valid': True,
 'rows': 500000,
 'credits_required': 500000,
 'ai_rows_required': 0}

## 3. Generate 500,000 rows

In [4]:
started = time.time()
big = generate(big_spec)
print(f"Ready in {time.time() - started:.0f} seconds")
summary(big)

Ready in 60 seconds


{'status': 'ready',
 'format': 'sql',
 'rows_generated': 500000,
 'rows_per_table': {'order_items': 319149,
  'customers': 42553,
  'products': 10638,
  'orders': 127660},
 'credits_charged': 500000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_cPMn19W8zl7KTZt0',
 'seed': 2026}

In [5]:
sql_file = download(big, "shop_500k.sql")
db_file = download(big, "shop_500k.sqlite", file="db")
{f.name: f"{f.stat().st_size / 1e6:.1f} MB" for f in (sql_file, db_file)}

{'shop_500k.sql': '25.5 MB', 'shop_500k.sqlite': '21.0 MB'}

## 4. Work with it

Half a million rows are comfortable in SQLite and pandas.

In [6]:
import sqlite3

import pandas as pd

con = sqlite3.connect(db_file)
pd.read_sql_query("""
    SELECT substr(order_date, 1, 7) AS month, COUNT(*) AS orders, ROUND(SUM(order_total), 2) AS revenue
    FROM orders WHERE status <> 'cancelled' GROUP BY month ORDER BY month""", con).head(8)

,month,orders,revenue
0,2025-01,2293,1687052.12
1,2025-02,2372,1838334.57
2,2025-03,3020,2167416.28
3,2025-04,3522,2606323.87
4,2025-05,4289,3152569.44
5,2025-06,4634,3368828.16
6,2025-07,5444,4057674.20
7,2025-08,5954,4402546.09


In [7]:
pd.read_sql_query("""
    SELECT orders_placed, COUNT(*) AS customers
    FROM (SELECT customer_id, COUNT(*) AS orders_placed FROM orders GROUP BY customer_id)
    GROUP BY orders_placed ORDER BY orders_placed LIMIT 8""", con)

,orders_placed,customers
0,1,11158
1,2,12867
2,3,8427
3,4,4537
4,5,2204
5,6,1058
6,7,651
7,8,389


The relationships hold at this size too:

In [8]:
checks = {
    "order items without an order": "SELECT COUNT(*) FROM order_items i LEFT JOIN orders o USING (order_id) WHERE o.order_id IS NULL",
    "orders without a customer": "SELECT COUNT(*) FROM orders o LEFT JOIN customers c USING (customer_id) WHERE c.customer_id IS NULL",
    "line_total is not quantity x unit_price": "SELECT COUNT(*) FROM order_items WHERE ABS(line_total - quantity * unit_price) > 0.011",
}
pd.DataFrame([(rule, con.execute(sql).fetchone()[0]) for rule, sql in checks.items()], columns=["rule", "rows breaking it"])

,rule,rows breaking it
0,order items without an order,0
1,orders without a customer,0
2,line_total is not quantity x unit_price,0


## Good to know

- One request can ask for up to 500,000 rows. For more, run the same `schema_id` again with a different `seed`.
- Large CSV downloads are best read in pieces: `pd.read_csv(path, chunksize=100_000)`.
- One dataset runs at a time per account; the helper above waits if another is still running.
- Files are kept for 30 days. The same `schema_id` and `seed` re-create the data at any time.